# Программирование на Python, БИ

## НИУ ВШЭ, 2025-26 учебный год

# Семинар 13. Regular Expressions

Сегодня мы с вами обсудим очень важную, широко используемую во всех разделах анализа данных и прокладывающую прямой мостик к разговору о Data Engineering тему — регулярные выражения.

~~Хотя, судя по вашим ГП, вы и так все большие специалисты в этом... А точнее, ваш GPT~~

## Введение

Регулярные выражения — или Regular Expressions, RegEx — в анализе данных и задачах обработки текста в целом (не обязательно в привязке напрямую к Python) — это функционал специальных выражений, представляющих собой последовательности символов, которые позволяют искать определенные совпадения в тексте.

Выражаясь более формально, они помогают найти подстроки некоторого вида в строке. Еще о регулярных выражениях можно думать как о шаблонах, в которые мы можем подставлять текст, и этот текст либо соответствует шаблону, либо нет. В самом простом случае в качестве регулярного выражения может использоваться обычная строка.

По следующей ссылке вы найдете подробную [документацию](https://docs.python.org/3/library/re.html). А вот здесь представлен очень хороший [CheatSheet](https://www.dataquest.io/wp-content/uploads/2019/03/python-regular-expressions-cheat-sheet.pdf). Рекомендуем сохранить последний и подглядывать в него, особенно поначалу, когда пишете программы с "регулярками".

In [1]:
# библиотека регулярных выражений в Python
import re

Итак, зачем же всё-таки нужны регулярные выражения? Почему нельзя обойтись обычными средствами Python? Попробуем разобраться на примере!

Возьмем две первых строки из известной считалки Агаты Кристи и попытаемся найти в них слово "один". Для этого можно воспользоваться оператором `in`. Он проверяет точное вхождение одной строки в другую и возвращает логическое значение: `True`, если вхождение есть, и `False` в противном случае.

In [2]:
string = 'Десять негритят отправились обедать, \
          Один поперхнулся, их осталось девять.'
# слово "Один" с заглавной буквы есть в строке
'Один' in string

True

In [3]:
# а если мы попытаемся поискать слово "один" с маленькой буквы, то оператор вернет False
'один' in string

False

Видно, что слово *Один* начинается с большой буквы. Что, если мы хотим найти в некоторой строке слово *Один* вне зависимости от регистра, то есть все слова типа *Один, один, ОДИН* и т.д.?

Эту задачу все еще можно решить без регулярных выражений: привести всю строку к нижнему регистру и искать слово *один*.

А что, если у нас будет текст подлиннее (например, полный текст считалки), и в нем необходимо найти все числительные от одного до десяти во всех падежах (один/одного/двое/двоим и т.д.)? В такой ситуации удобнее написать некоторый шаблон, чтобы не создавать длинный список слов с разными формами слов. Тут-то на помощь и придут регулярные выражения.

## Синтаксис

Теперь давайте разберемся с основными особенностями синтаксиса (но не забываем и про CheatSheet, упоминавшийся выше):

* Промежутки, заключенные в квадратные скобки, позволяют найти цифры или буквы разных алфавитов и разных регистров:

    `[0-9]` соответствует любой цифре
    
    `[A-Z]` соответствует любой заглавной букве английского алфавита
    
    `[a-z]` соответствует любой строчной букве английского алфавита
    
    `[А-Я]` и `[а-я]` — аналогично для букв русского алфавита (кроме буквы `ё/Ё` — ее нужно включать отдельно!)

* Для обозначения произвольного символа (кроме новой строки) используется  точка — `.`.

* Для цифр есть специальный символ `\d` (от *digit*) ≈ `[0-9]`.

  Добавление обратного слэша называется экранированием: так мы отмечаем, что ищем именно цифру, а не просто букву d.

* Для любого символа, кроме цифры, тоже есть специальный символ `\D` (от *digit*) ≈ `[^0-9]` (заглавная буква здесь отвечает за отрицание).

  Добавление обратного слэша называется экранированием: так мы отмечаем, что ищем именно цифру, а не просто букву d.

* Для пробела тоже существует свой символ — `\s` (от *space*) ≈ `[ \f\n\r\t\v]`. Этот символ соответствует ровно одному пробельному символу в тексте (пробел, табуляция, перенос строки и т.д.).

* Любой непробельный символ, обозначается как `\S` (заглавная буква здесь отвечает за отрицание).

* Для букв тоже существует свой символ — `\w` (от *word*) ≈ `[0-9a-zA-Zа-яА-ЯёЁ_]`. Любая буква (то, что может быть частью слова), а также цифры и _ .

* Любая не-буква, не-цифра и не подчёркивание, обозначается как `\W` (заглавная буква здесь отвечает за отрицание).



Кроме того, существуют специальные операторы-расширители — на которых, на самом деле, в первую очередь и строятся регулярные выражения:

* Знак `.` соответствует одному любому символу в строке. Так, регулярное выражение `x.x` "поймает" слова *хах* и *хех*.
* Знак `+` соответствует одному или более вхождению символа(ов), который стоит слева от `+`. Выражение `xa+` "поймает" слова *xa* и *хаааа*.
* Знак `*` соответствует нулю или более вхождениям символа, который стоит слева от `*`.  Выражение `xaх*`  "поймает" слова *xa* и *хах*.
* Знак `?` соответствует нулю или одному вхождению символа, который стоит слева от `?`.  Выражение `xa?`  "поймает" все последовательности *xa* и буквы *x*.



Но что, если у нас нет определенного шаблона, и нам надо вернуть набор символов из строки, отвечающий определенным правилам? Такая задача часто стоит при извлечении информации из строк. Это можно сделать, написав выражение с использованием специальных символов. Вот наиболее часто используемые из них:

- `\b` — Граница слова
- `[..]` — Один из символов в скобках (`[^..]` — любой символ, кроме тех, что в скобках)
- `\`	— Экранирование специальных символов (например, `\.` означает точку или `\+` — знак «плюс»)
- `^` и `$`	— Начало и конец строки соответственно
- `{n, m}` — От n до m вхождений (`{,m}` — от 0 до m)
- `a|b` —	Соответствует a или b
- `()` — Группирует выражение и возвращает найденный текст
- `\t`, `\n` — Символ табуляции, новой строки соответственно

## Использование RegEx в Python

В Python мы можем использовать встроенный модуль `re` для работы с регулярными выражениями.

Существует ряд команд, которые помогут нам в этом:

* `re.search(pattern, string)` — Ищет первое вхождение шаблона в строке и возвращает объект `match`. Чтобы вывести содержимое нужно использовать метод `.group()`.
* `re.findall(pattern, string)` — Находит все вхождения шаблона в строке и возвращает список совпадающих строк.
* `re.sub(pattern, repl, string)` — Заменяет все вхождения шаблона в строке строкой-заменителем и возвращает измененную строку.



Посмотрим примеры:

In [4]:
import re
st = 'hfjdsaklhgdsakl 543627 44y3t2ui 4tyu345673'

# добавляем r, чтобы Python читал это как raw string (не видя никаких спецсимволов)
re.findall(r'\d{1,3}',st)

['543', '627', '44', '3', '2', '4', '345', '673']

In [5]:
st = 'ёёёёёёёёабв'
re.search(r'[а-я]+',st).group()

'абв'

Здесь нашлось абв, потому что в диапазон а-я буква ё не входит!

In [6]:
st = 'Революция произошла в 1917 году'
re.search(r'\d+',st).group()

'1917'

In [7]:
st = 'Революция произошла в 1917 году'
re.search(r'\D+',st).group()

'Революция произошла в '

In [8]:
# Мы добежали до первого пробельного символа
st = 'Революция произошла в 1917 году 34'
re.findall(r'\d+',st)

['1917', '34']

Для разбора дальнейших символов в регулярных выражениях создадим небольшой набор слов (не очень осмысленный, но удобный):

     хах, хех, хаааа, xaxa

In [9]:
st = "хах, хех, хаааа, хаххххххха"
re.findall(r'хах*',st)

['хах', 'ха', 'хаххххххх']

In [10]:
re.findall(r'ха+',st)

['ха', 'хаааа', 'ха', 'ха']

In [11]:
st = "хах, хех, хаааа, хаха"
re.findall(r'хах*',st) #ха из хаааа, т.к * говорит нам что слева должно быть ха и справа 0 или бесконечно число х

['хах', 'ха', 'хах']

In [12]:
st = "хах, хех, хаа..аа, хаха 3456 456 56436743"
re.findall(r'\d{3}',st)

['345', '456', '564', '367']

Хорошо, а как быть, если с помощью регулярного выражения нужно найти подстроку, содержащую знаки препинания? Те же точки, вопросительные знаки, скобки?

Ответ — нужно их экранировать, то есть ставить перед ними `\`, например, `\.`, `\,`, `\?`. Этот символ будет сообщать Python, что нам нужен именно конкретный символ (точка, запятая, знак вопроса и др.).

Кроме того, как уже упоминалось выше, в регулярных выражениях можно явно задавать число повторений символов. Если мы знаем точное число символов, то его можно указать в фигурных скобках. Так, выражение `а{4}` будет соответствовать четырем буквам `a` подряд.

Если точное число повторений нам неизвестно, можно задать диапазон, указав начало и конец отрезка через запятую. Например, такое выражение позволит найти от двух до четырех букв `a` подряд: `a{2,4}`.

Если известен только левый или правый конец отрезка, то второй конец можно опустить: `a{2,}` (не менее двух) или `a{,4}` (не более 4).

В регулярных выражениях также можно использовать условие *или*. Например, возвращаясь к нашей "смеющейся" строке, если мы напишем выражение `x[а|е]х`,  оно поймает слова *хах* и *хех*, а вот вдруг появившийся *хох* не поймает.



In [13]:
st = "хах, хех, ?хаа..аа, хаха, хох"
re.findall(r'х[а|е]х',st)

['хах', 'хех', 'хах']

Создадим какой-нибудь незамысловатый текст с разными датами:

In [14]:
text = "12 ноября 2011 года произошло удивительное событие. А 13 ноября 2012 — еще удивительнее. Даже не будем \
говорить, что произошло 2 декабря 2011 года и 25 декабря 2012 года."
text

'12 ноября 2011 года произошло удивительное событие. А 13 ноября 2012 — еще удивительнее. Даже не будем говорить, что произошло 2 декабря 2011 года и 25 декабря 2012 года.'

In [15]:
spisok = re.findall(r"\d{1,2}\s[а-яё]+\s\d{2}12", text) # отдельно цифры
spisok

['13 ноября 2012', '25 декабря 2012']

Если забыли, что числа можно искать с помощью `\d`, можно задействовать промежуток (только не забудьте квадратные скобки):

In [16]:
re.findall("[0-9]", text)

['1',
 '2',
 '2',
 '0',
 '1',
 '1',
 '1',
 '3',
 '2',
 '0',
 '1',
 '2',
 '2',
 '2',
 '0',
 '1',
 '1',
 '2',
 '5',
 '2',
 '0',
 '1',
 '2']

А что, если мы хотим "ловить" не цифры, а числа, то есть последовательности из одной или более цифры?

Условию "один и более", как мы знаем, соответствует символ `+`. Попробуем.

In [17]:
re.findall(r"\d+", text) # отдельно числа

['12', '2011', '13', '2012', '2', '2011', '25', '2012']

Получилось! А если сочетания по 1-2 цифры (иногда с пробелом после)?

Тут нужен знак `.`, который отвечает ровно за один символ.

In [18]:
re.findall(r"\d.", text) # отдельно числа по 1-2 цифры

['12', '20', '11', '13', '20', '12', '2 ', '20', '11', '25', '20', '12']

Что будет, если мы воспользуемся знаком `?`? Он отвечает за наличие 0 или 1 символа, стоящего слева от регулярного выражения.

In [19]:
re.findall(r"\d?", text) # по 1 символу

['1',
 '2',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '2',
 '0',
 '1',
 '1',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '1',
 '3',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '2',
 '0',
 '1',
 '2',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '2',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '2',
 '0',
 '1',
 '1',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '2',
 '5',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '2',
 '0',
 '1',
 '2',
 '',
 '',
 '',
 '',
 '',
 '',
 '']

Получили какое-то безобразие. Но это безобразие оправдано: добавив `?` мы поставили условие, что в подстроке либо есть ровно одна цифра, либо ее нет. Поэтому мы и получили такой странный список.

Давайте рассмотрим теперь какую-нибудь задачу, где необходимо применить экранирование.

Пусть у нас есть некоторая строка с данными:

In [20]:
data = '20.05.1963, 55, 12.12.2000, 17, 15/15/1111'

И пусть нам необходимо выбрать из нее даты, записанные через точку. Напишем регулярное выражение, которое позволит это сделать, но перед этим вспомним, что точку нужно экранировать — ставить перед ней `\`, чтобы Python понимал, что мы ищем не один любой символ (`.`), а именно точку как знак препинания.

In [21]:
re.findall(r'\d{2}\.\d{2}\.\d{4}', data)

['20.05.1963', '12.12.2000']

In [22]:
re.findall(r"\d+\.\d+\.\d{4}", data) # готово

['20.05.1963', '12.12.2000']

In [23]:
re.findall(r'\d+/\d+/\d{4}', data)

['15/15/1111']

### Задание для самостоятельного решения (RegEx 1)

Напишите регулярное выражение, которое будет "ловить" все слова с основой *удивительн* в тексте.


In [24]:
text = "12 ноября 2011 года произошло преудивительнейшее событие. А 13 ноября 2012 - еще удивительнее. Даже не будем \
говорить, что произошло 2 декабря 2011 года и 25 декабря 2012 года."

In [25]:
# ваш код

### Задание для самостоятельного решения (RegEx 2)

Напишите регулярное выражение, которое будет из списка хештегов "ловить" только хештеги, начинающиеся с `#я не могу`.


In [26]:
tweets = ["#я не могу молчать", "#я не могу кричать", "#я не могу", "#я справлюсь", "я не могу молчать",
        "#я не могу жить", "#я все могу", "#с кем не бывает"]

In [27]:
# ваш код

### Задание для самостоятельного решения (RegEx 3)*

Напишите регулярное выражение, которое будет "ловить" все года в тексте.

Обратите внимание, в тексте могут быть элементы из 4 цифр, которые не являются годами.

In [28]:
text = "По имеющимся данным, в Екатеринбургской губернии на май 1916 года было занято 50611 военнопленных,\
из них 34194 на фабричных и заводских работах, 5731 на «казённых», 5060 на сельскохозяйственных,\
4145 на железнодорожных, 913 на городских и земских, 1953 на прочих. В 1899 г. произошло ужасное."
text

'По имеющимся данным, в Екатеринбургской губернии на май 1916 года было занято 50611 военнопленных,из них 34194 на фабричных и заводских работах, 5731 на «казённых», 5060 на сельскохозяйственных,4145 на железнодорожных, 913 на городских и земских, 568 на прочих.'

In [29]:
# ваш код

## Другие функции регулярных выражений

До этого мы по большей части работали с функцией `findall`, но в модуле `re` есть и другие полезные функции.

Вот наиболее часто используемые из них:

- `re.match()`
- `re.search()`
- `re.split()`
- `re.sub()`
- `re.compile()`

Рассмотрим их подробнее.

### Функция match

Данная функция — `re.match(pattern, string)` — ищет совпадения по заданному шаблону в начале строки.

Например, если мы вызовем метод `match()` на строке «Сидоров Иван Иванович» с шаблоном «Сидоров», то он завершится успешно. Однако если мы будем искать «Иван», то результат будет отрицательный. Давайте посмотрим на его работу:

In [30]:
result = re.match(r'\d+\w+', '1223ghjhgtyuiСидоров Иван Петрович')
print(result)

<re.Match object; span=(0, 20), match='1223ghjhgtyuiСидоров'>


Искомая подстрока найдена. Чтобы вывести ее содержимое, используем метод `group()`.

In [31]:
result.group()

'1223ghjhgtyuiСидоров'

Теперь попробуем найти «Иван» в данной строке. Поскольку строка начинается с фамилии, метод вернет `None`:

In [32]:
result = re.match(r'Иван', 'Сидоров Иван Петрович')
print(result)

None


Также есть методы `start()` и `end()` для того, чтобы узнать начальную и конечную позицию найденной строки.

In [33]:
result = re.match(r'Сидоров', 'Сидоров Иван Петрович')
print(result.start())
print(result.end())

0
7


Эти методы иногда очень полезны для работы со строками.

### Функция search

Данная функция — `re.search(pattern, string)` — отчасти похожа на `match()`, но она ищет совпадения не только в начале строки.

В отличие от предыдущего, `search()` вернет объект, если мы попытаемся найти «Иван».

In [34]:
result = re.search(r'Иван', 'Сидоров Иван Петрович')
result

<re.Match object; span=(8, 12), match='Иван'>

Метод `search()` ищет по всей строке, но возвращает только первое найденное совпадение.

### Функция split

Данная функция — `re.split(pattern, string, [maxsplit=0])` — разделяет строку по заданному шаблону.

In [35]:
result = re.split(r' ', 'Сидоров Иван Петрович')
result

['Сидоров', 'Иван', 'Петрович']

В примере мы разделили нашу строку по пробелам. Метод `split()` принимает также аргумент `maxsplit` со значением по умолчанию, равным 0. В данном случае он разделит строку столько раз, сколько возможно, но если указать этот аргумент, то разделение будет произведено не более указанного количества раз. Давайте посмотрим на примеры:

In [36]:
result = re.split(r' ', 'Сидоров Иван Петрович', maxsplit=1)
result

['Сидоров', 'Иван Петрович']

Мы установили параметр `maxsplit` равным 1, и в результате строка была разделена на две части вместо трех.

Приведем "боевой" пример. У нас есть некоторая ссылка и её нужно разобрать на параметры, которые потом нужно обработать. Здесь видны адрес сайта, судя по всему, тип страницы, название товарной позиции и какой-то непонятный id \
https://beru.ru/product/konditsioner-dlia-belia-aroma-rich-fairy-lion-0-43-l-paket/100583881004

In [37]:
site = 'https://beru.ru/product/konditsioner-dlia-belia-aroma-rich-fairy-lion-0-43-l-paket/100583881004'
result = re.split(r'\b/', site)
result

['https://beru.ru',
 'product',
 'konditsioner-dlia-belia-aroma-rich-fairy-lion-0-43-l-paket',
 '100583881004']

### Функция compile

Данная функция — `re.compile(pattern, flags=0)` — компилирует объект регулярного выражения для последующего использования.

Примеры: [Compilation flags](https://learnbyexample.github.io/py_regular_expressions/flags.html)

In [38]:
text = 'hfjklhgadsfk_7fjhsdkl867589065,bhjk4\n \
        5jfnkld45'
digitRegex = re.compile(r'[0-9]+') # Не менее 1 цифры

''' Сейчас мы их проверим...'''
digitRegex.findall('teуе132213xt')

['132213']

### Функция sub

Данная функция — `re.sub(pattern, repl, string)` — ищет шаблон в строке и заменяет его на указанную подстроку. Если шаблон не найден, строка остается неизменной.

In [39]:
result = re.sub(r'[;,]', ' ', 'Сидоров;Иван,Петрович')
result

'Сидоров Иван Петрович'

## Примеры задач с решением

Давайте рассмотрим еще несколько примеров.

### Задача 1. Извлечь слова на гласную букву

Задача — извлечь все слова, начинающиеся на гласную.

Для начала вернем все слова:

In [40]:
string = 'ОГО Десять негритят отправились обедать, \
          Один поперхнулся, их осталось девять.'
re.findall(r'\w+', string)

['ОГО',
 'Десять',
 'негритят',
 'отправились',
 'обедать',
 'Один',
 'поперхнулся',
 'их',
 'осталось',
 'девять']

А теперь — только те, которые начинаются на определенные буквы (используя `[]`):

In [41]:
re.findall(r'[аяоеуюыиэеАЯОЕУЮЫИЭ]\w+', string)

['ОГО',
 'есять',
 'егритят',
 'отправились',
 'обедать',
 'Один',
 'оперхнулся',
 'их',
 'осталось',
 'евять']

Что-то тут не так) Если слово не начинается с глаcной, оно было обрезано до первой гласной. Для того, чтобы убрать их, используем `\b` для обозначения границы слова:

In [42]:
re.findall(r'\b[аяоеуюыиэеАЯОЕУЮЫИЭ]\w+', string)

['ОГО', 'отправились', 'обедать', 'Один', 'их', 'осталось']

Получили искомый результат. Но что если нам нужно получить слова начинающиеся с согласной буквы? Мы, конечно, можем их всех перечислить как гласные, но это не оптимальный способ. Мы можем использовать `^` внутри квадратных скобок для инвертирования группы:

In [43]:
re.findall(r'\b[^аяоеуюыиэеАЯОЕУЮЫИЭ]\w+', string)

[' Десять',
 ' негритят',
 ' отправились',
 ' обедать',
 ' поперхнулся',
 ' осталось',
 ' девять']

В результат попали слова, «начинающиеся» с пробела. Уберем их, включив пробел в диапазон в квадратных скобках:

In [44]:
# слова, не начинающиеся с гласной или пробела
re.findall(r'\b[^аяоеуюыиэеАЯОЕУЮЫИЭ ]\w+', string)

['Десять', 'негритят', 'поперхнулся', 'девять']

### Задача 2. Проверить телефонный номер

Задача — проверить телефонный номер: номер должен быть длиной 10 знаков и начинаться с 7 или 8.

Очень актуальная, кстати, задача, и давайте решим её, используя регулярные выражения.

Для каждого номера мы проверям его по следующему паттерну: сначала стоит либо 7, либо 8 `[7-8]`, строго один раз `{1}`. Далее идет блок из любых цифр `[0-9]` длинной строго в 9 `{9}`. Ну и в конце мы проверяем длину "числа". Если все хорошо, помечаем номер как верный.`[0-9]` можно также заменить на `\d`, а `{1}` можно опустить. Потому что `[]` по умолчанию подразумевают наличие одного элемента из скобок.

In [45]:
tel = '8999999999'

if re.match(r'[7-8]{1}[0-9]{9}\b', tel) == None:
    print('Номер введен не корректно')
else:
    print('Все ок')

# result = re.match(r'[7-8]{1}\d{9}\b', tel)

Все ок


### Задача 3. Проверить номера автомобилей

В России применяются регистрационные знаки нескольких видов.
Общего в них то, что они состоят из цифр и букв. Причём используются только 12 букв кириллицы, \
имеющие графические аналоги в латинском алфавите — А, В, Е, К, М, Н, О, Р, С, Т, У и Х.

У частных легковых автомобилях номера — это буква, три цифры, две буквы, затем две или три цифры с кодом региона. У такси — две буквы, три цифры, затем две или три цифры с кодом региона. Есть также и другие виды, но в этой задаче они не понадобятся.

Создайте программу, которая будет проверять номер автомобиля и выдавать тип автомобиля: 'Private' для частных автомобилей, 'Taxi' для такси, 'Fail' для всех остальных.

In [ ]:
numbers = ['С227НА777','КУ22777','Т22В7477','М227К19У9','С227НА8777']
# правильные ответы
Answers = ['Private','Taxi','Fail','Fail','Fail']

In [ ]:
# ваш код

### Задача 4. Извлечь информацию из html-файла

Допустим, нам надо извлечь информацию из html-файла, заключенную между `<td>` и `</td>` (из таблицы), кроме первого столбца с номером. Также будем считать, что html-код содержится в строке.

Пример содержимого html-файла:

In [ ]:
test_str = "1NoahEmma2LiamOlivia3MasonSophia4JacobIsabella5WilliamAva6EthanMia7MichaelEmily"
test_str

'1NoahEmma2LiamOlivia3MasonSophia4JacobIsabella5WilliamAva6EthanMia7MichaelEmily'

Для лучшего понимания строки, распишем ее в другом виде:

1. NoahEmma
2. LiamOlivia
3. MasonSophia
4. ...

Сначала мы ищем число, которое будет у нас означать начало новой строки в таблице (стартовую позицию поиска в строке) `\d`. После этого мы указываем, что имя начинается с заглавной буквы `[A-Z]`. Получим число и первую букву имени.

In [ ]:
re.findall(r'\d[A-Z]', test_str)

['1N', '2L', '3M', '4J', '5W', '6E', '7M']

Далее забираем основную часть имени `[a-z]`, там уже строчные буквы. Используем модификатор `+`, чтобы взять все буквы.

In [ ]:
re.findall(r'\d[A-Z][a-z]+', test_str)

['1Noah', '2Liam', '3Mason', '4Jacob', '5William', '6Ethan', '7Michael']

Но у нас в начале есть номер и имя с фамилией слеплены вместе. Сначала разберемся с номером: `()` позволяют нам указать, что мы будем выводить `([A-Z][a-z]+)`, число останется вне вывода и будет использоваться только для определения позиции следующего применения паттерна.

In [ ]:
re.findall(r'\d([A-Z][a-z]+)', test_str)

['Noah', 'Liam', 'Mason', 'Jacob', 'William', 'Ethan', 'Michael']

Теперь займемся фамилией. По условию задачи мы знаем, что фамилия тоже начинается с заглавной буквы. Поэтому мы укажем второй (аналогичный имени) паттерн и занесем его в `()` для отдельного вывода

In [ ]:
re.findall(r'\d([A-Z][a-z]+)([A-Z][a-z]+)', test_str)

[('Noah', 'Emma'),
 ('Liam', 'Olivia'),
 ('Mason', 'Sophia'),
 ('Jacob', 'Isabella'),
 ('William', 'Ava'),
 ('Ethan', 'Mia'),
 ('Michael', 'Emily')]

Как видно по выводу, фамилия теперь отображается как второй элемент первого вывода паттерна.

### Задача 5. Поиск однокоренных слов

Нужно в тексте найти все однокоренные слова. Будем выполнять на данной скороговорке:

Рыла свинья белорыла, тупорыла; полдвора рылом изрыла, вырыла, подрыла.

In [ ]:
pig = "Рыла свинья белорыла, тупорыла; полдвора рылом изрыла, вырыла, подрыла."
pig

'Рыла свинья белорыла, тупорыла; полдвора рылом изрыла, вырыла, подрыла.'

Корень может быть как в начале слова, так и где-то в середине. Поэтому учтем это. Сначала у нас могут быть буквы `[а-яА-Я]` длиной от 0 до бесконечности `*`

In [ ]:
re.findall(r'[а-яА-Я]*', "Рыла свинья белорыла, тупорыла; полдвора рылом изрыла, вырыла, подрыла.")

['Рыла',
 '',
 'свинья',
 '',
 'белорыла',
 '',
 '',
 'тупорыла',
 '',
 '',
 'полдвора',
 '',
 'рылом',
 '',
 'изрыла',
 '',
 '',
 'вырыла',
 '',
 '',
 'подрыла',
 '',
 '']

Нам попались все слова и пробелы, так как `*`. Далее будем искать наш корень. Нам нужно точное совпадение с `рыл` или `Рыл` для случая с началом предложения `(?:рыл|Рыл)`. `|` говорит нам о выборе между `рыл` и `Рыл`, т.е. подойдет любой из них.

Что делает `(?:)` ? Этот символ помогает нам вернуть последовательность полностью. Выше мы уже видели, что, то, что последовательность в скобках соответствует формату вывода. В этом случае `(?:)` это меняет: если последовательность символов подходит ВСЕМУ шаблону, то оно и будет возвращено функцией findall.

Но если мы уберем ?:, то любая последовательность подходящая под внутренний паттерн скобок будет выведена.

In [ ]:
re.findall(r'[а-яА-Я]*(?:рыл|Рыл)', "Рыла свинья белорыла, тупорыла; полдвора рылом изрыла, вырыла, подрыла.")

['Рыл', 'белорыл', 'тупорыл', 'рыл', 'изрыл', 'вырыл', 'подрыл']

In [ ]:
re.findall(r'[а-яА-Я]*(рыл|Рыл)', "Рыла свинья белорыла, тупорыла; полдвора рылом изрыла, вырыла, подрыла.")

['Рыл', 'рыл', 'рыл', 'рыл', 'рыл', 'рыл', 'рыл']

Есть начало слова и его корень. Осталось добавить окончание. Все аналогично началу слова:

In [ ]:
re.findall(r'[а-яА-Я]*(?:рыл|Рыл)[а-яА-Я]*', "Рыла свинья белорыла, тупорыла; полдвора рылом изрыла, вырыла, подрыла.")

['Рыла', 'белорыла', 'тупорыла', 'рылом', 'изрыла', 'вырыла', 'подрыла']

### Задача 6. Проверка пароля

На сайте нужно вывести сообщение, если пароль не валиден. \
Обычно пароль должен содержать не менее 8 символов, не менее одной заглавной буквы, не менее одной строчной буквы и, опционально, символ

In [ ]:
passwordText = 'fbbE4gfhdsjk'

charRegex = re.compile(r'(\w{8,})')  # Не менее 8 символов (тут не считаются знаки препинания)
lowerRegex = re.compile(r'[a-z]+') # Не менее 1 маленькой буквы
upperRegex = re.compile(r'[A-Z]+')# Не менее 1 большой буквы
digitRegex = re.compile(r'[0-9]+') # Не менее 1 цифры

''' Сейчас мы их проверим...'''
if charRegex.findall(passwordText) == []:
    print('Пароль должен содержать 8 символов')
elif lowerRegex.findall(passwordText)== []:
    print('Пароль должен содержать хотя бы одну маленькую букву')
elif upperRegex.findall(passwordText)==[]:
    print('Пароль должен содержать хотя бы одну большую букву')
elif digitRegex.findall(passwordText)==[]:
    print('Пароль должен содержать хотя бы одну цифру')
else:
    print('Ваш пароль надежен. Хорошая работа!')

Ваш пароль надежен. Хорошая работа!


## Группирующие скобки

Если в шаблоне регулярного выражения встречаются скобки `(...)` без `?:`, то они становятся группирующими. В match-объекте по каждой такой группе можно получить ту же информацию, что и по всему шаблону. А именно часть подстроки, которая соответствует `(...)`, а также индексы начала и окончания в исходной строке.

In [46]:
import re
pattern = r'\s*([А-Яа-яЁё]+)(\d+)\s*'

string = r'---   Опять45   ---'

match = re.search(pattern, string)

print(f'Найдена подстрока >{match[0]}< с позиции {match.start(0)} до {match.end(0)}')

print(f'Группа букв >{match[1]}< с позиции {match.start(1)} до {match.end(1)}')
print(f'Группа цифр >{match[2]}< с позиции {match.start(2)} до {match.end(2)}')
###
# -> Найдена подстрока >   Опять45   < с позиции 3 до 16
# -> Группа букв >Опять< с позиции 6 до 11
# -> Группа цифр >45< с позиции 11 до 13

Найдена подстрока >   Опять45   < с позиции 3 до 16
Группа букв >Опять< с позиции 6 до 11
Группа цифр >45< с позиции 11 до 13


In [47]:
match.group(2)

'45'

In [48]:
s = 'It\'s been a long time'
match = re.search(r'(\s\w)(\w\w)', s)
match[0]

' bee'

## Продвинутые позиционные шаблоны

Продвинутые позиционные шаблоны позволяют осуществлять заглядывание вперед и назад.

Следующие шаблоны применяются в основном в тех случаях, когда нужно уточнить, что должно идти непосредственно перед или после шаблона, но при этом
не включать найденное в match-объект.

* `(?=...)` lookahead assertion, соответствует каждой позиции, сразу после которой начинается соответствие шаблону

* `(?!...)` negative lookahead assertion, соответствует каждой позиции, сразу после которой НЕ может начинаться шаблон
* `(?<=...)` positive lookbehind assertion, соответствует каждой позиции, которой может заканчиваться шаблон. Длина шаблона должна быть фиксированной, то есть abc и a|b — это ОК, а a* и a{2,3} — нет.

* `(?<!...)` negative lookbehind assertion, соответствует каждой позиции, которой НЕ может заканчиваться шаблон

In [ ]:
string = 'КарлIV, КарлIX, КарлV, КарлVI, КарлVII, КарлVIII,ЛюдовикIX, \
ЛюдовикVI, ЛюдовикVII, ЛюдовикVIII, ЛюдовикX, ..., ЛюдовикXVIII, ФилиппI, \
ФилиппII, ФилиппIII, ФилиппIV, ФилиппV, ФилиппVI'

# Людовик, после которого первые два символа == VI
re.findall(r'Людовик(?=VI)\w{2,4}',string)

['ЛюдовикVI', 'ЛюдовикVII', 'ЛюдовикVIII']

In [ ]:
string = 'КарлIV, КарлIX, КарлV, КарлVI, КарлVII, КарлVIII,ЛюдовикIX, \
ЛюдовикVI, ЛюдовикVII, ЛюдовикVIII, ЛюдовикX, ..., ЛюдовикXVIII, ФилиппI, \
ФилиппII, ФилиппIII, ФилиппIV, ФилиппV, ФилиппVI'
# Людовик, после которого первые два символа != VI

re.findall(r'Людовик(?!VI)\w{1,4}',string)

['ЛюдовикIX', 'ЛюдовикX', 'ЛюдовикXVII']

In [ ]:
string = 'КарлIV, КарлIX, КарлV, КарлVI, КарлVII, КарлVIII,ЛюдовикIX, \
ЛюдовикVI, ЛюдовикVII, ЛюдовикVIII, ЛюдовикX, ..., ЛюдовикXVIII, ФилиппI, \
ФилиппII, ФилиппIII, ФилиппIV, ФилиппV, ФилиппVI'
# цифра, начинающаяся на VI, только если перед ней стоит Людовик

re.findall(r'\w*(?<=Людови.)VI\w{,2}',string)

['ЛюдовигVI', 'ЛюдовикVII', 'ЛюдовикVIII']

In [ ]:
import re
string = 'КарлIV, КарлIX, КарлV, КарлVI, КарлVII, КарлVIII,ЛюдовигIX, \
ЛюдовикVI, ЛюдовикVII, ЛюдовикVIII, ЛюдовикX, ..., ЛюдовикXVIII, ФилиппI, \
ФилиппII, ФилиппIII, ФилиппIV, ФилиппV, ФилиппVI'

# цифра, начинающаяся на VI, только если перед ней НЕ стоит Людовик
re.findall(r'[а-яА-Я]+(?<!Людови.)VI\w{,2}',string)

['КарлVI', 'КарлVII', 'КарлVIII', 'ФилиппVI']

## Дополнительные упражения

### Задача 7. Достать из строки имена-фамилии на русском языке

In [ ]:
users = 'Василий Зайцев, Erwin König, Людмила Павличенко, Josef Allerberger, Matthäus Hetzenauer, Александр Башлачёв'

In [ ]:
# ваш код

### Задача 8. Выделить из строки email'ы

Например, из такой: `'iawpghnube1206@gmail.com\r\n+79151489999 (telegram @vasiiesal) test.tewst2@subsubdomain.subdomain.domain.ru.!'`

In [ ]:
x = 'iawpgh-----nube1206@gmail.com\r\n+79151489999 (telegram @vasiiesal) test.tewst2@subsubdomain.subdomain.domain.ru.!'
re.findall(r'\b', x) # строки в выводе пустые, потому что не указали какой паттерн искать

['',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '']

Далее идет блок в `[ ]`, в котором как раз определяем паттерн, который ищем, в нашем случае это `S`, где `\S` любой непробельный символ. НО это только один символ, попадающий в эту маску. Для того чтобы получить неограниченную длинной последовательность, мы добавим + к квадратным скобкам.

Получим такие вот подстроки:
- iawpghnube1206@gmail.com
- 79151489999
- telegram
- vasiiesal)
- test.tewst2!@subsubdomain.subdomain.domain.ru.!

In [ ]:
re.findall(r'\b[\S]+', x)

['iawpgh-----nube1206@gmail.com',
 '79151489999',
 'telegram',
 'vasiiesal)',
 'test.tewst2@subsubdomain.subdomain.domain.ru.!']

`@` показывает нам, что далее нужно найти 1 символ `@`. Что даст следующие подстроки:

- iawpghnube1206@
- test.tewst2!@

In [ ]:
re.findall(r'\b[\S]+@', x)

['iawpgh-----nube1206@', 'test.tewst2@']

После `@` всегда идет домен. Как говорилось выше он может иметь несколько уровней. Поэтому мы снова ищем последовательность из букв, цифр и спец знаков, причем данная последовательность встречается от 1 и более раз `{1,}`

Получим такие строки:
- iawpghnube1206@gmail.com'
- test.tewst2!@subsubdomain.subdomain.domain.ru.!

In [ ]:
re.findall(r'\b[\S]+@[\S]{1,}', x)

['iawpgh-----nube1206@gmail.com',
 'test.tewst2@subsubdomain.subdomain.domain.ru.!']

Все выглядит хорошо, кроме того что мы захватили с собой лишние знаки. Однако мы знаем, что почта всегда заканчивается точкой и доменной зоной. Попробуем, это учесть. Укажем, что мы хотим ровно одну точку `\.` и неограниченное количество букв `\w+`.В данном случае мы прямо указываем на то что должны быть только буквы, цифры и спецсимволы не могу быть в доменной зоне. Получим:
- iawpghnube1206@gmail.com
- test.tewst2!@subsubdomain.subdomain.domain.ru

In [ ]:
re.findall(r'\b[\w\d\S]+@[\w\d\S]{1,}\.\w+', x)

['iawpgh-----nube1206@gmail.com',
 'test.tewst2@subsubdomain.subdomain.domain.ru']

## Заключение

Сегодня мы с вами познакомились с функциями из модуля `re`. Дополнительно про них можно почитать в официальной [документации](https://docs.python.org/3/library/re.html).

Кроме того, есть очень хороший ресурс [regex101.com](https://regex101.com), который позволяет скопировать нужный текст и в интерактивном режиме следить, какие совпадения находятся при изменении регулярного выражения, введенного в отдельном окне (не забудьте поставить галочку Python в разделе FLAVOR слева).

## Самостоятельная работа

### Сопоставление даты

Предположим, у нас есть строка, содержащая дату в формате `dd/mm/yyyy`. Мы хотим извлечь дату из строки и проверить, является ли она действительной (вдруг там 95/32/2930).

Напишите функцию `extract_date(text)`, которая принимает на вход строку `text` и возвращает строку, содержащую извлеченную дату в формате `yyyy-mm-dd`, если дата действительна, или пустую строку, если дата не найдена или не действительна.


Используйте функцию `re.search()` для поиска шаблона, который соответствует дате в строке.<br>
Используйте функцию `datetime.datetime.strptime()` для преобразования извлеченной строки даты в объект datetime.<br>
Используйте функцию `datetime.datetime.strftime()`, чтобы преобразовать объект даты обратно в строку в нужном формате.

In [ ]:
# ваш код

In [ ]:
text = "Сегодняшняя дата: 05/06/2023"
extract_date(text)

'2023-06-05'

### Извлечение URL-адресов

Предположим, у нас есть строка, содержащая один или несколько URL-адресов. Мы хотим извлечь URL-адреса из строки и сохранить их в списке.

Напишите функцию `extract_urls(text)`, которая принимает на вход строку `text` и возвращает список строк, содержащих извлеченные URL.

Используйте функцию `re.findall()` для поиска всех вхождений шаблона, который соответствует URL в строке.<br>
Используйте regex-шаблон, который соответствует URL в различных форматах, например http://example.com, https://www.example.com и www.example.com.

In [ ]:
# ваш код

In [ ]:
text = "Посетите мой веб-сайт: http://www.example.com, а также http://google.com и www.learnonline.hse.ru"
extract_urls(text)

['http://www.example.com,', 'http://google.com', 'www.learnonline.hse.ru']

### Определение платежной системы банковской карты по БИН

Напишите программу, используя регулярные выражения, которая будет определять, какая платежная система у введенного номера банковской карточки (ровно 16 цифр).

Небольшой справочник по первым цифрам БИН:
* 2 -Мир
* 3 - American Express, JCB International, Diners Club
    * 30,36,38-Diners Club
    * 31,35-JCB International
    * 34,37-American Express
* 4 - VISA
* 5 - MasterCard, Maestro
    * 50,56,57,58-Maestro
    * 51,52,53,54,55-MasterCard
* 6- Maestro, China UnionPay, Discover
    * 60-Discover
    * 62 - China UnionPay
    * 63, 67 - Maestro

In [ ]:
# ваш код